<a href="https://colab.research.google.com/github/ojaswip2828/ETL_Pipeline_Tests/blob/main/ETL_Pipeline_Tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ETL Pipeline + Tests
**Python + pandas + pytest** — fetches from a REST API, transforms with pandas, saves to CSV, includes pytest unit tests.

Run cells top to bottom. The last cell runs the full pytest suite inline using `ipytest`.

## 1. Install dependencies

In [1]:
!pip install -q requests pandas pytest ipytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.5 MB/s eta 0:00:00


## 2. Imports

In [2]:
import logging
from typing import Any

import pandas as pd
import requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

DEFAULT_URL = "https://jsonplaceholder.typicode.com/users"
DEFAULT_OUT = "output.csv"

## 3. Extract

In [3]:
def extract(url: str, timeout: int = 10) -> list[dict[str, Any]]:
    """Fetch JSON data from a REST API endpoint.

    Raises:
        requests.HTTPError: if the response status is not 2xx.
        requests.RequestException: on connection/timeout errors.
    """
    logger.info("Fetching data from %s", url)
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    data = response.json()

    if isinstance(data, dict):
        data = [data]
    if not isinstance(data, list):
        raise ValueError(f"Unexpected API response shape: {type(data)}")

    logger.info("Fetched %d records", len(data))
    return data

## 4. Transform

In [4]:
def transform(records: list[dict[str, Any]]) -> pd.DataFrame:
    """Flatten nested JSON and clean the data into a tidy DataFrame.

    - Flattens nested dicts (e.g. address.city -> address_city)
    - Drops fully-empty columns
    - Strips whitespace from string columns
    - Deduplicates rows
    """
    if not records:
        return pd.DataFrame()

    df = pd.json_normalize(records, sep="_")
    df = df.dropna(axis=1, how="all")

    str_cols = df.select_dtypes(include="object").columns
    for col in str_cols:
        df[col] = df[col].apply(lambda v: v.strip() if isinstance(v, str) else v)

    df = df.drop_duplicates().reset_index(drop=True)

    logger.info("Transformed data into DataFrame with shape %s", df.shape)
    return df

## 5. Load

In [5]:
def load(df: pd.DataFrame, out_path: str) -> None:
    """Write the DataFrame to CSV."""
    df.to_csv(out_path, index=False)
    logger.info("Saved %d rows to %s", len(df), out_path)


def run_pipeline(url: str = DEFAULT_URL, out_path: str = DEFAULT_OUT) -> pd.DataFrame:
    records = extract(url)
    df = transform(records)
    load(df, out_path)
    return df

## 6. Run it

In [6]:
df = run_pipeline()
df.head()

,id,name,username,email,phone,website,address_street,address_suite,address_city,address_zipcode,address_geo_lat,address_geo_lng,company_name,company_catchPhrase,company_bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


In [7]:

try:
    from google.colab import files
    files.download(DEFAULT_OUT)
except ImportError:
    print(f"Saved to {DEFAULT_OUT} — not running in Colab, so download() was skipped.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Unit tests (pytest, run inline via ipytest)
Network calls are mocked with `unittest.mock`, so these run offline and deterministically.

In [8]:
import ipytest
ipytest.autoconfig()

In [9]:
%%ipytest -v

from unittest.mock import MagicMock, patch
import pytest


@pytest.fixture
def sample_records():
    return [
        {
            "id": 1,
            "name": " Leanne Graham ",
            "email": "leanne@example.com",
            "address": {"city": "Gwenborough", "zipcode": "92998-3874"},
        },
        {
            "id": 2,
            "name": "Ervin Howell",
            "email": "ervin@example.com",
            "address": {"city": "Wisokyburgh", "zipcode": "90566-7771"},
        },
    ]


def _mock_response(json_data, status_code=200):
    mock_resp = MagicMock()
    mock_resp.status_code = status_code
    mock_resp.json.return_value = json_data
    if status_code >= 400:
        mock_resp.raise_for_status.side_effect = requests.HTTPError(f"{status_code} error")
    else:
        mock_resp.raise_for_status.return_value = None
    return mock_resp


class TestExtract:
    @patch("__main__.requests.get")
    def test_extract_returns_list(self, mock_get, sample_records):
        mock_get.return_value = _mock_response(sample_records)
        result = extract("https://fake-api.test/users")
        assert result == sample_records
        mock_get.assert_called_once_with("https://fake-api.test/users", timeout=10)

    @patch("__main__.requests.get")
    def test_extract_wraps_single_dict_in_list(self, mock_get):
        mock_get.return_value = _mock_response({"id": 1, "name": "Solo"})
        result = extract("https://fake-api.test/user/1")
        assert result == [{"id": 1, "name": "Solo"}]

    @patch("__main__.requests.get")
    def test_extract_raises_on_http_error(self, mock_get):
        mock_get.return_value = _mock_response({}, status_code=404)
        with pytest.raises(requests.HTTPError):
            extract("https://fake-api.test/missing")

    @patch("__main__.requests.get")
    def test_extract_raises_on_unexpected_shape(self, mock_get):
        mock_get.return_value = _mock_response("not a list or dict")
        with pytest.raises(ValueError):
            extract("https://fake-api.test/weird")

    @patch("__main__.requests.get", side_effect=requests.ConnectionError("network down"))
    def test_extract_propagates_connection_error(self, mock_get):
        with pytest.raises(requests.ConnectionError):
            extract("https://fake-api.test/users")


class TestTransform:
    def test_transform_returns_dataframe(self, sample_records):
        result = transform(sample_records)
        assert isinstance(result, pd.DataFrame)
        assert len(result) == 2

    def test_transform_flattens_nested_fields(self, sample_records):
        result = transform(sample_records)
        assert "address_city" in result.columns
        assert "address" not in result.columns

    def test_transform_strips_whitespace(self, sample_records):
        result = transform(sample_records)
        assert result.loc[0, "name"] == "Leanne Graham"

    def test_transform_drops_duplicates(self, sample_records):
        doubled = sample_records + [sample_records[0]]
        result = transform(doubled)
        assert len(result) == 2

    def test_transform_empty_input_returns_empty_dataframe(self):
        result = transform([])
        assert isinstance(result, pd.DataFrame)
        assert result.empty


class TestLoad:
    def test_load_writes_csv(self, tmp_path, sample_records):
        result = transform(sample_records)
        out_file = tmp_path / "out.csv"
        load(result, str(out_file))
        assert out_file.exists()
        reloaded = pd.read_csv(out_file)
        assert len(reloaded) == len(result)


class TestRunPipeline:
    @patch("__main__.requests.get")
    def test_full_pipeline_end_to_end(self, mock_get, tmp_path, sample_records):
        mock_get.return_value = _mock_response(sample_records)
        out_file = tmp_path / "result.csv"
        result = run_pipeline(url="https://fake-api.test/users", out_path=str(out_file))
        assert out_file.exists()
        assert len(result) == 2

======================================= test session starts ========================================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collected 12 items

t_6a4dd275992b4a5e84155c8a48d8ad8a.py ............                                           [100%]

======================================== 12 passed in 0.10s ========================================
